In [1]:
"""
Reinforcement Learning — All Four Algorithms on the Same 2x2 Grid
==================================================================

Grid:
    +------+------+
    |  S0  |  S1  |
    | start|      |
    +------+------+
    |  S2  | GOAL |
    |      | +10  |
    +------+------+

Rules:
    - Actions: 0=UP, 1=DOWN, 2=LEFT, 3=RIGHT
    - Off-grid → stay in place (still pay -1)
    - Step cost = -1, GOAL = +10 (episode ends)
    - States: S0=0, S1=1, S2=2, GOAL=3 (terminal)

Algorithms covered:
    1. Q-learning  (model-free, off-policy,  value-based, tabular)
    2. SARSA       (model-free, on-policy,   value-based, tabular)
    3. DQN         (model-free, off-policy,  value-based, neural network)
    4. REINFORCE   (model-free, on-policy,   policy-based, neural network)
"""

import numpy as np
import random
from collections import deque

# ─────────────────────────────────────────────────────────────────────────────
# ENVIRONMENT  (shared by all four algorithms)
# ─────────────────────────────────────────────────────────────────────────────
S0, S1, S2, GOAL = 0, 1, 2, 3
UP, DOWN, LEFT, RIGHT = 0, 1, 2, 3
ACTION_NAMES = ["UP", "DOWN", "LEFT", "RIGHT"]
STATE_NAMES  = ["S0", "S1", "S2", "GOAL"]

# transitions[state][action] = next_state
TRANSITIONS = {
    S0: {UP: S0,   DOWN: S2,   LEFT: S0,   RIGHT: S1},
    S1: {UP: S1,   DOWN: GOAL, LEFT: S0,   RIGHT: S1},
    S2: {UP: S0,   DOWN: S2,   LEFT: S2,   RIGHT: GOAL},
}

def env_step(state, action):
    """Take one step. Returns (next_state, reward, done)."""
    next_state = TRANSITIONS[state][action]
    if next_state == GOAL:
        return GOAL, 10.0, True
    return next_state, -1.0, False

def env_reset():
    """Start a new episode at S0."""
    return S0

def encode(state):
    """One-hot encode state for neural networks. GOAL → all zeros (terminal)."""
    v = np.zeros(3, dtype=np.float32)
    if state < 3:
        v[state] = 1.0
    return v

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. Q-LEARNING
# ─────────────────────────────────────────────────────────────────────────────
def run_qlearning(n_episodes=300, alpha=0.5, gamma=0.9,
                  epsilon=0.1, verbose=True):
    """
    Q-Learning: model-free, off-policy, value-based.

    Key idea:
        Q(s,a) <- Q(s,a) + alpha * [r + gamma * max Q(s',.) - Q(s,a)]

    Action selection: epsilon-greedy on Q-table
    Update:          always uses MAX Q at next state (off-policy)
    """
    print("\n" + "="*60)
    print("1. Q-LEARNING")
    print("="*60)

    # ── Q-table: 3 states x 4 actions, all zeros ──────────────────────────
    Q = np.zeros((3, 4))

    def choose_action(state):
        """Epsilon-greedy on the Q-table."""
        if np.random.random() < epsilon:
            return np.random.randint(4)                   # EXPLORE: random
        return int(np.argmax(Q[state]))                   # EXPLOIT: best Q

    rewards_per_episode = []



# simply based on the studied formula
    for episode in range(n_episodes):
        state = env_reset()
        total_reward = 0
        done = False

        while not done:
            # ── Step 1: choose action (epsilon-greedy) ─────────────────────
            action = choose_action(state)

            # ── Step 2: take action, observe result ────────────────────────
            next_state, reward, done = env_step(state, action)
            total_reward += reward

            # ── Step 3: Q-learning update ──────────────────────────────────
            # future term = MAX Q at next state (off-policy)
            best_next = 0.0 if done else np.max(Q[next_state])

            # TD error = target - current estimate
            td_error = reward + gamma * best_next - Q[state, action]

            # nudge Q toward the target
            Q[state, action] += alpha * td_error

            state = next_state

        rewards_per_episode.append(total_reward)

    # ── Results ────────────────────────────────────────────────────────────
    if verbose:
        print("\nLearned Q-table:")
        print(f"{'':6}" + "".join(f"{a:>8}" for a in ACTION_NAMES))
        for s in [S0, S1, S2]:
            print(f"{STATE_NAMES[s]:6}" + "".join(f"{Q[s,a]:8.2f}" for a in range(4)))

        print("\nOptimal policy (best action per state):")
        for s in [S0, S1, S2]:
            best = ACTION_NAMES[int(np.argmax(Q[s]))]
            print(f"  {STATE_NAMES[s]} -> {best}")

        avg = np.mean(rewards_per_episode[-50:])
        print(f"\nAverage reward (last 50 episodes): {avg:.2f}")

    return Q

In [3]:
# 1. Q-learning
q_table = run_qlearning(n_episodes=300)


1. Q-LEARNING

Learned Q-table:
            UP    DOWN    LEFT   RIGHT
S0        6.20    4.36    6.09    8.00
S1        7.93   10.00    6.20    7.94
S2        6.17   -0.50    2.09    0.00

Optimal policy (best action per state):
  S0 -> RIGHT
  S1 -> DOWN
  S2 -> UP

Average reward (last 50 episodes): 8.70


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. SARSA
# ─────────────────────────────────────────────────────────────────────────────
def run_sarsa(n_episodes=300, alpha=0.5, gamma=0.9,
              epsilon=0.1, verbose=True):
    """
    SARSA: model-free, on-policy, value-based.

    Key idea:
        Q(s,a) <- Q(s,a) + alpha * [r + gamma * Q(s',a') - Q(s,a)]     

    Action selection: epsilon-greedy on Q-table
    Update:          uses Q of the ACTUAL NEXT ACTION chosen by epsilon-greedy
                     (on-policy — learns about the policy it follows)

    vs Q-learning:   Q-learning uses MAX Q(s',.) (best possible)
                     SARSA uses Q(s', a') (actual next action)
                     Difference only appears when a' is not the best action
    """
    print("\n" + "="*60)
    print("2. SARSA")
    print("="*60)

    # ── Q-table: 3 states x 4 actions, all zeros ──────────────────────────
    Q = np.zeros((3, 4))

    def choose_action(state):
        """Epsilon-greedy on the Q-table."""
        if np.random.random() < epsilon:
            return np.random.randint(4)                   # EXPLORE: random
        return int(np.argmax(Q[state]))                   # EXPLOIT: best Q

    rewards_per_episode = []

    for episode in range(n_episodes):
        state  = env_reset()
        action = choose_action(state)                     # choose FIRST action
        total_reward = 0
        done = False

        while not done:
            # ── Step 1: take action already chosen ─────────────────────────
            next_state, reward, done = env_step(state, action)
            total_reward += reward

            # ── Step 2: choose NEXT action (needed BEFORE update) ──────────
            # This is the key SARSA difference — must pick a' before updating
            if done:
                next_action = 0          # terminal, doesn't matter
            else:
                next_action = choose_action(next_state)   # epsilon-greedy picks a'

            # ── Step 3: SARSA update ───────────────────────────────────────
            # future term = Q of ACTUAL next action (on-policy)
            next_q = 0.0 if done else Q[next_state, next_action]

            # TD error
            td_error = reward + gamma * next_q - Q[state, action]

            # nudge Q toward the target
            Q[state, action] += alpha * td_error

            # carry chosen action forward
            state  = next_state
            action = next_action

        rewards_per_episode.append(total_reward)

    # ── Results ────────────────────────────────────────────────────────────
    if verbose:
        print("\nLearned Q-table:")
        print(f"{'':6}" + "".join(f"{a:>8}" for a in ACTION_NAMES))
        for s in [S0, S1, S2]:
            print(f"{STATE_NAMES[s]:6}" + "".join(f"{Q[s,a]:8.2f}" for a in range(4)))

        print("\nOptimal policy (best action per state):")
        for s in [S0, S1, S2]:
            best = ACTION_NAMES[int(np.argmax(Q[s]))]
            print(f"  {STATE_NAMES[s]} -> {best}")

        avg = np.mean(rewards_per_episode[-50:])
        print(f"\nAverage reward (last 50 episodes): {avg:.2f}")

        print("\nKey difference from Q-learning:")
        print("  SARSA uses Q(s',a') — Q of the actual next action chosen by epsilon-greedy")
        print("  Q-learning uses max Q(s',.) — always the best possible next Q")
        print("  When epsilon-greedy picks a suboptimal action, SARSA gives lower (safer) values")

    return Q

In [5]:

# 2. SARSA
sarsa_table = run_sarsa(n_episodes=300)




2. SARSA

Learned Q-table:
            UP    DOWN    LEFT   RIGHT
S0        4.90    5.13    5.22    6.82
S1        7.69   10.00    2.35    7.99
S2        1.76   -0.75    1.09   10.00

Optimal policy (best action per state):
  S0 -> RIGHT
  S1 -> DOWN
  S2 -> RIGHT

Average reward (last 50 episodes): 8.68

Key difference from Q-learning:
  SARSA uses Q(s',a') — Q of the actual next action chosen by epsilon-greedy
  Q-learning uses max Q(s',.) — always the best possible next Q
  When epsilon-greedy picks a suboptimal action, SARSA gives lower (safer) values


In [14]:
%pip install torch

Defaulting to user installation because normal site-packages is not writeable
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
    --------------------------------------- 1.8/123.0 MB 10.1 MB/s eta 0:00:13
   - -------------------------------------- 4.5/123.0 MB 11.7 MB/s eta 0:00:11
   -- ------------------------------------- 6.8/123.0 MB 12.0 MB/s eta 0:00:10
   --- ------------------------------------ 9.7/123.0 MB 12.3 MB/s eta 0:00:10
   ---- ----------------------------------- 13.1/123.0 MB 13.0 MB/s eta 0:00:09
   ----- ---------------------------------- 17.3/123.0 MB 14.5 MB/s eta 0:00:08
   ------- -------------------------------- 22.5/123.0 MB 16.0 MB/s eta 0:00:07
   -------- ------------------------------- 26.5/123.0 MB 16.6 MB/s eta 0:00:06
   ---------- ----------------------------- 31.2/123.0 MB 17.7 MB/s eta 0:00:06
   -------


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\ashwi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. DQN
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import random
from collections import deque

try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False

def run_dqn(n_episodes=300, gamma=0.9, lr=0.01,
            epsilon_start=1.0, epsilon_min=0.05,
            epsilon_decay=0.97, batch_size=32,
            target_update_every=20, verbose=True):
    """
    DQN: model-free, off-policy, value-based, neural network.

    Key idea:
        Same as Q-learning BUT replaces the Q-table with a neural network.
        Two extra tricks needed because of the network:
            1. Experience replay buffer  (break correlated data)
            2. Target network           (stable training target)

    Action selection: epsilon-greedy on main_net (same as Q-learning)
    Update:          train main_net toward (r + gamma * max target_net(s'))
    """
    if not TORCH_AVAILABLE:
        print("\nDQN skipped — PyTorch not installed.")
        return None

    print("\n" + "="*60)
    print("3. DQN — Deep Q-Network")
    print("="*60)

    # ── Q-Network: 3 inputs (one-hot state) -> 4 outputs (Q per action) ───
    class QNetwork(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(3, 16), nn.ReLU(),    # 3 inputs -> 16 hidden
                nn.Linear(16, 4)                 # 16 hidden -> 4 Q-values
            )
        def forward(self, x):
            return self.net(x)

    # ── Two networks: main (trains every step) + target (frozen copy) ─────
    main_net   = QNetwork()
    target_net = QNetwork()
    target_net.load_state_dict(main_net.state_dict())  # start identical

    optimizer = torch.optim.Adam(main_net.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()

    # ── Replay buffer ──────────────────────────────────────────────────────
    buffer  = deque(maxlen=2000)
    epsilon = epsilon_start
    step_count = 0

    def choose_action(state):
        """Epsilon-greedy — reads main_net (no gradient needed)."""
        if np.random.random() < epsilon:
            return np.random.randint(4)                        # EXPLORE
        with torch.no_grad():
            q = main_net(torch.FloatTensor(encode(state)))
        return int(torch.argmax(q))                            # EXPLOIT

    rewards_per_episode = []

    for episode in range(n_episodes):
        state = env_reset()
        total_reward = 0
        done = False

        while not done:
            # ── Step 1: choose action (epsilon-greedy on main_net) ─────────
            action = choose_action(state)

            # ── Step 2: take action, observe result ────────────────────────
            next_state, reward, done = env_step(state, action)
            total_reward += reward

            # ── Step 3: store in replay buffer ─────────────────────────────
            buffer.append((state, action, reward, next_state, done))
            state = next_state
            step_count += 1

            # ── Steps 4-6: train if buffer has enough experiences ──────────
            if len(buffer) >= batch_size:

                # Step 4: sample RANDOM batch (breaks correlation)
                batch = random.sample(buffer, batch_size)
                states, actions, rewards_, next_states, dones = zip(*batch)

                # convert to tensors
                states_t      = torch.FloatTensor(
                                    np.array([encode(s) for s in states]))
                next_states_t = torch.FloatTensor(
                                    np.array([encode(s) for s in next_states]))
                actions_t     = torch.LongTensor(actions)
                rewards_t     = torch.FloatTensor(rewards_)
                dones_t       = torch.FloatTensor(dones)

                # Step 5: compute target using FROZEN target_net
                with torch.no_grad():
                    max_next_q = target_net(next_states_t).max(dim=1)[0]
                    # (1 - done) zeroes out future term for terminal states
                    targets = rewards_t + gamma * max_next_q * (1 - dones_t)

                # Step 6: compute prediction from main_net, compute loss
                # gather picks the Q-value for the action actually taken
                current_q = main_net(states_t)\
                              .gather(1, actions_t.unsqueeze(1)).squeeze(1)

                loss = loss_fn(current_q, targets)


# adam backpropagation algorithm optimizer
# relu activation function
                # backprop — update main_net weights
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            # ── Step 7: refresh target_net periodically ────────────────────
            if step_count % target_update_every == 0:
                target_net.load_state_dict(main_net.state_dict())

        # decay epsilon after each episode
        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        rewards_per_episode.append(total_reward)

    # ── Results ────────────────────────────────────────────────────────────
    if verbose:
        print("\nLearned Q-values (from main_net):")
        print(f"{'':6}" + "".join(f"{a:>8}" for a in ACTION_NAMES))
        for s in [S0, S1, S2]:
            with torch.no_grad():
                q = main_net(torch.FloatTensor(encode(s))).numpy()
            print(f"{STATE_NAMES[s]:6}" + "".join(f"{q[a]:8.2f}" for a in range(4)))

        print("\nOptimal policy (best action per state):")
        for s in [S0, S1, S2]:
            with torch.no_grad():
                q = main_net(torch.FloatTensor(encode(s))).numpy()
            best = ACTION_NAMES[int(np.argmax(q))]
            print(f"  {STATE_NAMES[s]} -> {best}")

        avg = np.mean(rewards_per_episode[-50:])
        print(f"\nAverage reward (last 50 episodes): {avg:.2f}")

        print("\nKey differences from Q-learning:")
        print("  Q-table replaced by neural network (generalizes to unseen states)")
        print("  Experience replay buffer breaks correlation in training data")
        print("  Target network provides stable training target (no moving goalpost)")

    return main_net


In [7]:
# 3. DQN
dqn_net = run_dqn(n_episodes=300)




DQN skipped — PyTorch not installed.


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. REINFORCE
# ─────────────────────────────────────────────────────────────────────────────
def run_reinforce(n_episodes=500, gamma=0.9, lr=0.01, verbose=True):
    """
    REINFORCE: model-free, on-policy, policy-based, neural network.

    Key idea:
        Don't learn Q-values — directly optimize the POLICY.
        theta <- theta + alpha * G_t * grad log pi(a_t|s_t)

    Action selection: SAMPLE from policy network probabilities
                      (no epsilon-greedy — exploration is built in via sampling)
    Update:          after FULL EPISODE using actual return G_t
                     (Monte Carlo — no bootstrapping)
    Network output:  PROBABILITIES via softmax (not Q-values)
    """
    if not TORCH_AVAILABLE:
        print("\nREINFORCE skipped — PyTorch not installed.")
        return None

    print("\n" + "="*60)
    print("4. REINFORCE")
    print("="*60)

    # ── Policy network: 3 inputs -> softmax -> 4 action probabilities ──────
    class PolicyNetwork(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(3, 16), nn.ReLU(),    # 3 inputs -> 16 hidden
                nn.Linear(16, 4),               # 16 hidden -> 4 logits
                nn.Softmax(dim=-1)              # logits -> probabilities (sum=1)
            )
        def forward(self, x):
            return self.net(x)

    # ONE network only — no target network needed (no Bellman bootstrapping)
    policy_net = PolicyNetwork()
    optimizer  = torch.optim.Adam(policy_net.parameters(), lr=lr)

    rewards_per_episode = []

    for episode in range(n_episodes):
        state = env_reset()
        done  = False

        # ── Phase 1: play full episode, collect data ───────────────────────
        episode_states    = []
        episode_actions   = []
        episode_rewards   = []
        episode_log_probs = []

        while not done:
            # encode state and get action probabilities
            state_t = torch.FloatTensor(encode(state))

            with torch.no_grad():
                probs = policy_net(state_t)          # softmax probabilities

            # SAMPLE from probabilities — exploration built in!
            # (no epsilon-greedy needed)
            action = torch.multinomial(probs, 1).item() #// GIVES THE ACTION WITH THE HIGHEST PROBABILITY (EXPLOIT)

            # save log probability of the chosen action (needed for loss)
            log_prob = torch.log(policy_net(state_t)[action]) #//just the log of the probability of the action taken


            print(policy_net(state_t)) # 


            # take action
            next_state, reward, done = env_step(state, action)

            # record this step
            episode_states.append(state)
            episode_actions.append(action)
            episode_rewards.append(reward)
            episode_log_probs.append(log_prob)

            state = next_state

        total_reward = sum(episode_rewards)
        rewards_per_episode.append(total_reward)

        # ── Phase 2: compute returns G_t (backwards from end) ─────────────
        returns = []
        G = 0.0
        for r in reversed(episode_rewards):
            G = r + gamma * G
            returns.insert(0, G)               # insert at front (reverse order)

        returns_t = torch.FloatTensor(returns)

        # optional: normalize returns to reduce variance
        # returns_t = (returns_t - returns_t.mean()) / (returns_t.std() + 1e-8)

        # ── Phase 3: compute loss and update policy network ────────────────
        # loss = -G_t * log pi(a_t|s_t)   for each step
        # negative because we MAXIMIZE reward but torch MINIMIZES loss
        policy_loss = []
        for log_prob, G_t in zip(episode_log_probs, returns_t):
            policy_loss.append(-log_prob * G_t)

        loss = torch.stack(policy_loss).sum()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # ── Phase 4: discard episode data (on-policy — cannot reuse) ───────
        # episode_states, episode_actions, etc. go out of scope

    # ── Results ────────────────────────────────────────────────────────────
    if verbose:
        print("\nLearned policy (action probabilities per state):")
        print(f"{'':6}" + "".join(f"{a:>8}" for a in ACTION_NAMES))
        for s in [S0, S1, S2]:
            with torch.no_grad():
                probs = policy_net(torch.FloatTensor(encode(s))).numpy()
            print(f"{STATE_NAMES[s]:6}" + "".join(f"{probs[a]:8.3f}" for a in range(4)))

        print("\nOptimal policy (highest probability action per state):")
        for s in [S0, S1, S2]:
            with torch.no_grad():
                probs = policy_net(torch.FloatTensor(encode(s))).numpy()
            best = ACTION_NAMES[int(np.argmax(probs))]
            print(f"  {STATE_NAMES[s]} -> {best}  (prob={np.max(probs):.3f})")

        avg = np.mean(rewards_per_episode[-50:])
        print(f"\nAverage reward (last 50 episodes): {avg:.2f}")

        print("\nKey differences from DQN:")
        print("  Network outputs PROBABILITIES (softmax) not Q-values")
        print("  Actions chosen by SAMPLING from probs (no epsilon-greedy)")
        print("  Updates happen AFTER FULL EPISODE using actual return G_t")
        print("  NO replay buffer (on-policy — episode data discarded after use)")
        print("  NO target network (no Bellman bootstrapping)")

    return policy_net

In [9]:
# 4. REINFORCE
reinforce_net = run_reinforce(n_episodes=500)


REINFORCE skipped — PyTorch not installed.


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# INFERENCE — how to use a trained model
# ─────────────────────────────────────────────────────────────────────────────
def run_inference(model, algorithm_name):
    """
    Show how to use a trained model for deployment.
    No training — just forward pass and pick action.
    """
    print(f"\n--- Inference: {algorithm_name} ---")
    state = env_reset()
    done  = False
    path  = [STATE_NAMES[state]]
    total_reward = 0

    while not done:
        if algorithm_name in ["Q-learning", "SARSA"]:
            # Table: just look up the row and take argmax
            action = int(np.argmax(model[state]))

        elif algorithm_name == "DQN":
            # Network: encode -> forward pass -> argmax Q-values
            with torch.no_grad():
                q = model(torch.FloatTensor(encode(state)))
            action = int(torch.argmax(q))

        elif algorithm_name == "REINFORCE":
            # Network: encode -> forward pass -> softmax -> argmax probs
            with torch.no_grad():
                probs = model(torch.FloatTensor(encode(state)))
            action = int(torch.argmax(probs))   # greedy at inference

        next_state, reward, done = env_step(state, action)
        path.append(f"--({ACTION_NAMES[action]})--> {STATE_NAMES[next_state]}")
        total_reward += reward
        state = next_state

    print(f"  Path: {'  '.join(path)}")
    print(f"  Total reward: {total_reward}")


In [11]:


# Inference — deploy each trained model
print("\n" + "="*60)
print("INFERENCE (deployment) — greedy path from S0 to GOAL")
print("="*60)
run_inference(q_table,        "Q-learning")
run_inference(sarsa_table,    "SARSA")
if dqn_net:
    run_inference(dqn_net,    "DQN")
if reinforce_net:
    run_inference(reinforce_net, "REINFORCE")


INFERENCE (deployment) — greedy path from S0 to GOAL

--- Inference: Q-learning ---
  Path: S0  --(RIGHT)--> S1  --(DOWN)--> GOAL
  Total reward: 9.0

--- Inference: SARSA ---
  Path: S0  --(RIGHT)--> S1  --(DOWN)--> GOAL
  Total reward: 9.0


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# COMPARISON SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
def print_comparison():
    print("\n" + "="*60)
    print("ALGORITHM COMPARISON SUMMARY")
    print("="*60)
    rows = [
        ("Family",          "Value-based",   "Value-based", "Value-based",       "Policy-based"),
        ("On/Off policy",   "Off-policy",    "On-policy",   "Off-policy",        "On-policy"),
        ("Future term",     "max Q(s',·)",   "Q(s',a')",    "max target_net(s')", "G_t (actual return)"),
        ("Exploration",     "epsilon-greedy","epsilon-greedy","epsilon-greedy",   "Sample from probs"),
        ("Update timing",   "Every step",    "Every step",  "Every step",        "End of episode"),
        ("Neural network",  "No (table)",    "No (table)",  "Yes (Q-values)",    "Yes (probabilities)"),
        ("Replay buffer",   "No",            "No",          "Yes",               "No"),
        ("Target network",  "No",            "No",          "Yes",               "No"),
        ("Softmax output",  "No",            "No",          "No",                "Yes"),
        ("Uses Bellman",    "Yes (max)",     "Yes (actual)", "Yes (max)",         "No"),
    ]
    header = f"{'Property':<20} {'Q-learning':<15} {'SARSA':<15} {'DQN':<20} {'REINFORCE':<15}"
    print(header)
    print("-" * 85)
    for row in rows:
        print(f"{row[0]:<20} {row[1]:<15} {row[2]:<15} {row[3]:<20} {row[4]:<15}")


In [13]:


# Summary comparison
print_comparison()



ALGORITHM COMPARISON SUMMARY
Property             Q-learning      SARSA           DQN                  REINFORCE      
-------------------------------------------------------------------------------------
Family               Value-based     Value-based     Value-based          Policy-based   
On/Off policy        Off-policy      On-policy       Off-policy           On-policy      
Future term          max Q(s',·)     Q(s',a')        max target_net(s')   G_t (actual return)
Exploration          epsilon-greedy  epsilon-greedy  epsilon-greedy       Sample from probs
Update timing        Every step      Every step      Every step           End of episode 
Neural network       No (table)      No (table)      Yes (Q-values)       Yes (probabilities)
Replay buffer        No              No              Yes                  No             
Target network       No              No              Yes                  No             
Softmax output       No              No              No         